In [2]:
!pip install mediapipe==0.10.35 opencv-python

In [5]:
!wget https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task

--2026-05-19 03:16:55--  https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.26.207, 74.125.141.207, 142.251.107.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.26.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7819105 (7.5M) [application/octet-stream]
Saving to: ‘hand_landmarker.task’

hand_landmarker.tas 100%[===================>]   7.46M  --.-KB/s    in 0.1s    

2026-05-19 03:16:55 (76.2 MB/s) - ‘hand_landmarker.task’ saved [7819105/7819105]



In [6]:
import os
import torch
import tarfile
import cv2
import numpy as np
import pandas as pd
#import mediapipe as mp
import matplotlib.pyplot as plt

In [22]:
# STEP 1: Import the necessary modules.

#mediapipe 0.10.35, most recent one compatable with any python version that's self respecting (since 2023)
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# STEP 2: Create an HandLandmarker object.

# Install hand_landmarker.task from https://ai.google.dev/edge/mediapipe/solutions/vision/hand_landmarker/index#models
# Under Models click the link for hand marker, put that file in the same directory as this ipynb notebook
base_options = python.BaseOptions(model_asset_path='/kaggle/working/hand_landmarker.task')

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    
    # equivalent to static_image_mode=True
    running_mode=vision.RunningMode.IMAGE,

    # equivalent to max_num_hands=1
    num_hands=1,

    # equivalent to min_detection_confidence=0.5
    min_hand_detection_confidence=0.5
)
detector = vision.HandLandmarker.create_from_options(options)


W0000 00:00:1779162701.997212     309 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779162702.009746     308 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [9]:
from tqdm import tqdm

SEQUENCE_LENGTH = 37
NUM_LANDMARKS = 21
FEATURES_PER_FRAME = NUM_LANDMARKS * 3  # x, y, z = 63

# Function Nghi made

# Image path - > img input
# update mediapipe
#rgb image
def extract_landmarks_from_image(image_array):

    # Image handling

    if image_array is None:
        return np.zeros(FEATURES_PER_FRAME)

    # Mediapipe Stuff
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=image_array
    )
    
    detection_result = detector.detect(mp_image)

    #drawn_result = draw_landmarks_on_image(image_array, detection_result)

    #plt.imshow(drawn_result)
    #plt.show()

    if detection_result.hand_landmarks:
        hand_landmarks = detection_result.hand_landmarks[0]

        landmarks = []
        for lm in hand_landmarks:
            landmarks.extend([lm.x, lm.y, lm.z])

        return np.array(landmarks)

    else:
        return np.zeros(FEATURES_PER_FRAME)

In [29]:
def process_frame_folder(folder_path, sequence_length=SEQUENCE_LENGTH):
    frame_files = sorted(os.listdir(folder_path))

    # Keep only image files
    frame_files = [
        f for f in frame_files
        if f.lower().endswith(('.jpg'))
    ]

    # Take first 37 frames
    frame_files = frame_files[:sequence_length]

    
    sequence = []

    for frame in frame_files:
        frame_path = os.path.join(folder_path, frame)
        image = cv2.imread(frame_path)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        landmarks = extract_landmarks_from_image(image_rgb)
        sequence.append(landmarks)

    # If less than 37 frames, pad with zeros
    while len(sequence) < sequence_length:
        sequence.append(np.zeros(FEATURES_PER_FRAME))

    return np.array(sequence)

In [30]:
DATASET_PATH = "/kaggle/input/datasets/toxicmender/20bn-jester/Train"

all_folders = sorted([
    f for f in os.listdir(DATASET_PATH)
    if os.path.isdir(os.path.join(DATASET_PATH, f))
])

len(all_folders)
subset_folders = all_folders

In [31]:
csv_path = "/kaggle/input/datasets/toxicmender/20bn-jester/Train.csv"

df = pd.read_csv(csv_path)
label_dict = dict(zip(df['video_id'].astype(str), df['label_id']))
df

,video_id,label,frames,label_id,shape,format
0,1,Doing other things,37,0,"(100, 176)",JPEG
1,3,Pushing Two Fingers Away,37,6,"(100, 176)",JPEG
2,6,Drumming Fingers,37,1,"(100, 176)",JPEG
3,11,Sliding Two Fingers Down,37,10,"(100, 176)",JPEG
4,14,Pushing Hand Away,37,5,"(100, 176)",JPEG
...,...,...,...,...,...,...
50415,148084,No gesture,37,2,"(100, 176)",JPEG
50416,148085,Drumming Fingers,37,1,"(100, 176)",JPEG
50417,148088,Sliding Two Fingers Up,37,13,"(100, 176)",JPEG
50418,148090,Swiping Left,37,16,"(100, 176)",JPEG


In [43]:
result = df.set_index("label_id")["label"].to_dict()
result = sorted(result.items())
for idx, desc in result:
    print(f"{idx}: {desc}")


0: Doing other things
1: Drumming Fingers
2: No gesture
3: Pulling Hand In
4: Pulling Two Fingers In
5: Pushing Hand Away
6: Pushing Two Fingers Away
7: Rolling Hand Backward
8: Rolling Hand Forward
9: Shaking Hand
10: Sliding Two Fingers Down
11: Sliding Two Fingers Left
12: Sliding Two Fingers Right
13: Sliding Two Fingers Up
14: Stop Sign
15: Swiping Down
16: Swiping Left
17: Swiping Right
18: Swiping Up
19: Thumb Down
20: Thumb Up
21: Turning Hand Clockwise
22: Turning Hand Counterclockwise
23: Zooming In With Full Hand
24: Zooming In With Two Fingers
25: Zooming Out With Full Hand
26: Zooming Out With Two Fingers


In [40]:
import os
import numpy as np
import psutil
from tqdm import tqdm

os.makedirs("processedDataset", exist_ok=True)

chunk_size = 1000
chunk_index = 1

X_chunk = []
y_chunk = []

process = psutil.Process(os.getpid())

for i, folder_name in enumerate(tqdm(subset_folders)):

    folder_path = os.path.join(DATASET_PATH, folder_name)

    sequence = process_frame_folder(folder_path)
    label_index = label_dict[folder_name]

    X_chunk.append(sequence)
    y_chunk.append(label_index)

    # RAM monitor every 100 steps
    if i % 100 == 0:
        mem = process.memory_info().rss / 1024 / 1024
        avail = psutil.virtual_memory().available / 1024 / 1024
        print(f"[{i}] RAM used: {mem:.1f} MB | available: {avail:.1f} MB")

    # Save chunk
    if (i + 1) % chunk_size == 0:

        np.save(f"processedDataset/X_chunk_{chunk_index}.npy",
                np.array(X_chunk, dtype=object))
        np.save(f"processedDataset/y_chunk_{chunk_index}.npy",
                np.array(y_chunk))

        print(f"Saved chunk {chunk_index}")

        X_chunk.clear()
        y_chunk.clear()

        chunk_index += 1

  0%|          | 4/50420 [00:04<15:06:16,  1.08s/it]

Saved chunk 1
